# Day 18 — Advanced Patterns: Date Logic + Text
**Estimated time:** 60–75 min
**Datasets:** `hr_headcount.csv`, `cost_center_actuals.csv`

## Learning Objectives
- Perform date arithmetic in SQLite using `julianday()` and `strftime()`
- Calculate fiscal year/period offsets and employee tenure
- Use `GROUP_CONCAT` and conditional aggregation (`SUM(CASE WHEN ...)`)

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
# Load all CSVs
inv   = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
cc    = pd.read_csv(f"{DATA_DIR}/cost_center_actuals.csv")
hr    = pd.read_csv(f"{DATA_DIR}/hr_headcount.csv")
bw    = pd.read_csv(f"{DATA_DIR}/bw_sales_kpis.csv")

# Normalize column names
for df in [inv, sales, cc, hr, bw]:
    df.columns = [c.strip().upper() for c in df.columns]

# In-memory DuckDB — register pandas DataFrames as tables
con = duckdb.connect()
con.register("MATERIALS",   inv)
con.register("SALES",       sales)
con.register("COST_CENTER", cc)
con.register("HR",          hr)
con.register("BW_SALES",    bw)

def q(sql):
    return con.sql(sql).df()

# Verify
for tbl, df in [("MATERIALS",inv),("SALES",sales),("COST_CENTER",cc),("HR",hr),("BW_SALES",bw)]:
    n = q(f"SELECT COUNT(*) AS n FROM {tbl}").iloc[0,0]
    print(f"  {tbl:15s}: {n:,} rows")


In [ ]:
# -- Date arithmetic: employee tenure in years and months --
result = q(
    """
    SELECT
        PERNR,
        ENAME,
        HIRE_DATE,
        COALESCE(TERM_DATE, date('now')) AS effective_end_date,
        ROUND((julianday(COALESCE(TERM_DATE, date('now'))) - julianday(HIRE_DATE)) / 365.25, 2) AS tenure_years,
        CAST((julianday(COALESCE(TERM_DATE, date('now'))) - julianday(HIRE_DATE)) / 30.44 AS INT) AS tenure_months
    FROM HR
    WHERE HIRE_DATE IS NOT NULL
    ORDER BY tenure_years DESC
    LIMIT 15
    """
)
display(result)

In [ ]:
# -- strftime: extract year, month, quarter from date strings --
result = q(
    """
    SELECT
        HIRE_DATE,
        strftime('%Y', HIRE_DATE)                          AS hire_year,
        strftime('%m', HIRE_DATE)                          AS hire_month,
        CASE CAST(strftime('%m', HIRE_DATE) AS INT)
            WHEN 1  THEN 'Q1' WHEN 2  THEN 'Q1' WHEN 3  THEN 'Q1'
            WHEN 4  THEN 'Q2' WHEN 5  THEN 'Q2' WHEN 6  THEN 'Q2'
            WHEN 7  THEN 'Q3' WHEN 8  THEN 'Q3' WHEN 9  THEN 'Q3'
            ELSE 'Q4'
        END AS hire_quarter,
        COUNT(*) AS hires
    FROM HR
    WHERE HIRE_DATE IS NOT NULL
    GROUP BY hire_year, hire_month
    ORDER BY hire_year DESC, hire_month
    LIMIT 20
    """
)
display(result)

In [ ]:
# -- Fiscal year calculation (fiscal year starts Oct 1) --
result = q(
    """
    SELECT
        ERDAT,
        strftime('%Y', ERDAT) AS cal_year,
        strftime('%m', ERDAT) AS cal_month,
        CASE
            WHEN CAST(strftime('%m', ERDAT) AS INT) >= 10
                THEN CAST(strftime('%Y', ERDAT) AS INT) + 1
            ELSE CAST(strftime('%Y', ERDAT) AS INT)
        END AS fiscal_year,
        CASE
            WHEN CAST(strftime('%m', ERDAT) AS INT) >= 10
                THEN CAST(strftime('%m', ERDAT) AS INT) - 9
            ELSE CAST(strftime('%m', ERDAT) AS INT) + 3
        END AS fiscal_period
    FROM SALES
    WHERE ERDAT IS NOT NULL
    LIMIT 10
    """
)
display(result)

In [ ]:
# -- Tenure buckets with avg salary per bucket per org unit --
result = q(
    """
    WITH employee_tenure AS (
        SELECT
            PERNR,
            ENAME,
            ORGEH,
            ORGTX,
            SALARY,
            ROUND((julianday(COALESCE(TERM_DATE, date('now'))) - julianday(HIRE_DATE)) / 365.25, 2) AS tenure_years
        FROM HR WHERE HIRE_DATE IS NOT NULL AND SALARY IS NOT NULL
    ),
    bucketed AS (
        SELECT
            ORGTX,
            CASE
                WHEN tenure_years < 1  THEN '< 1 Year'
                WHEN tenure_years < 3  THEN '1-3 Years'
                WHEN tenure_years < 7  THEN '3-7 Years'
                WHEN tenure_years < 15 THEN '7-15 Years'
                ELSE '15+ Years'
            END AS tenure_bucket,
            SALARY
        FROM employee_tenure
    )
    SELECT
        tenure_bucket,
        ORGTX,
        COUNT(*)          AS headcount,
        ROUND(AVG(SALARY), 2) AS avg_salary,
        ROUND(MIN(SALARY), 2) AS min_salary,
        ROUND(MAX(SALARY), 2) AS max_salary
    FROM bucketed
    GROUP BY tenure_bucket, ORGTX
    ORDER BY tenure_bucket, avg_salary DESC
    """
)
display(result)

In [ ]:
# -- GROUP_CONCAT: list materials per plant (string aggregation) --
result = q(
    """
    SELECT
        WERKS,
        COUNT(DISTINCT MATNR) AS n_materials,
        GROUP_CONCAT(SUBSTR(MATNR, 1, 8), ', ') AS sample_materials
    FROM MATERIALS
    GROUP BY WERKS
    ORDER BY n_materials DESC
    LIMIT 10
    """
)
display(result)

In [ ]:
# -- Conditional aggregation: actual vs plan metrics in one row per cost center --
result = q(
    """
    SELECT
        KOSTL,
        KTEXT,
        GJAHR,
        SUM(CASE WHEN PERIOD <= 6 THEN ACTUAL_AMT ELSE 0 END) AS h1_actual,
        SUM(CASE WHEN PERIOD > 6  THEN ACTUAL_AMT ELSE 0 END) AS h2_actual,
        SUM(CASE WHEN ACTUAL_AMT > PLAN_AMT THEN 1 ELSE 0 END) AS periods_over_budget,
        SUM(CASE WHEN ACTUAL_AMT <= PLAN_AMT THEN 1 ELSE 0 END) AS periods_on_budget,
        ROUND(SUM(ACTUAL_AMT) / NULLIF(SUM(PLAN_AMT), 0) * 100, 2) AS total_budget_utilization_pct
    FROM COST_CENTER
    GROUP BY KOSTL, KTEXT, GJAHR
    ORDER BY periods_over_budget DESC
    LIMIT 15
    """
)
display(result)

---
## Daily Prompt

**For each employee, calculate their tenure in years and months. Group employees into tenure buckets and show average salary per bucket per department (ORGTX).**

The query above is a starting point. Extend it:
1. Add a `salary_vs_avg_ratio` column: each employee's salary / department average salary
2. Identify the tenure bucket with the highest average salary per department
3. Flag employees whose salary is > 30% above their department average as `HIGH_EARNER`

```python
# YOUR SOLUTION
result = q(
    """
    WITH tenure_data AS (
        -- YOUR SOLUTION: tenure calculation
    ),
    dept_avg AS (
        -- YOUR SOLUTION: avg salary per department
    )
    SELECT
        t.*,
        d.avg_dept_salary,
        -- YOUR SOLUTION: salary_vs_avg_ratio
        -- YOUR SOLUTION: HIGH_EARNER flag
    FROM tenure_data t
    JOIN dept_avg d ON t.ORGTX = d.ORGTX
    """
)
display(result)
```

> **Hint:** `salary_vs_avg_ratio = ROUND(SALARY * 1.0 / NULLIF(avg_dept_salary, 0), 2)`
> HIGH_EARNER: `CASE WHEN SALARY > avg_dept_salary * 1.30 THEN 'YES' ELSE 'NO' END`